In [ ]:
# Google Drive mount is only needed on Colab.
# On VS Code, your files are already on local disk — skip this cell.
# from google.colab import drive
# drive.mount('/content/drive')

In [28]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras import regularizers
REG = regularizers.l2(1e-4)

# ── 1. CUSTOM LAYERS (Bypassing Keras Save Bugs) ───────────────────────
class TileHorizon(layers.Layer):
    """Expands a single 2D spatial map into multiple identical future frames."""
    def __init__(self, horizon, **kwargs):
        super().__init__(**kwargs)
        self.horizon = horizon
    def call(self, x):
        return tf.tile(tf.expand_dims(x, 1), [1, self.horizon, 1, 1, 1])
    def get_config(self):
        return {**super().get_config(), 'horizon': self.horizon}

class StackHorizon(layers.Layer):
    """Stacks a 4D spatial tensor across the time dimension."""
    def __init__(self, horizon, **kwargs):
        super().__init__(**kwargs)
        self.horizon = horizon
    def call(self, x):
        return tf.stack([x] * self.horizon, axis=1)
    def get_config(self):
        return {**super().get_config(), 'horizon': self.horizon}

class ReduceMeanTime(layers.Layer):
    """Replaces Lambda(lambda t: tf.reduce_mean(t, axis=1)).
    Needed because Keras 3 blocks Lambda deserialization by default.
    """
    def call(self, x):
        return tf.reduce_mean(x, axis=1)
    def get_config(self):
        return super().get_config()

# ── 3. THE U-NET ARCHITECTURE ───────────────────────────────────────────
def build_cnn_lstm_unet(lookback=7, h=31, w=31, n_feats=5, horizon=3):
    inp = layers.Input(shape=(lookback, h, w, n_feats))

    # --- ENCODER (Spatial Feature Extraction) ---
    # Encoder — add kernel_regularizer to every Conv2D
    s1 = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(inp)
    s1 = layers.TimeDistributed(layers.BatchNormalization())(s1)
    s1 = layers.TimeDistributed(
        layers.Conv2D(64, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(s1)
    s1 = layers.TimeDistributed(layers.BatchNormalization())(s1)

    x = layers.TimeDistributed(layers.MaxPooling2D(2, padding='same'))(s1)

    s2 = layers.TimeDistributed(
        layers.Conv2D(128, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(x)
    s2 = layers.TimeDistributed(layers.BatchNormalization())(s2)

    # Bottleneck — increase Dropout from 0.3 to 0.4
    x = layers.ConvLSTM2D(128, kernel_size=3, padding='same',
                           return_sequences=False,recurrent_dropout=0.1)(s2)
    x = layers.Dropout(0.4)(x)   # was 0.3
    # x shape: (batch, 16, 16, 128)

    # 2. Duplicate that map into 3 future frames
    x = StackHorizon(horizon)(x)
    # x shape: (batch, 3, 16, 16, 128)

    # 3. Predict how the storm moves across those 3 frames
    x = layers.ConvLSTM2D(128, kernel_size=3, padding='same', return_sequences=True,recurrent_dropout=0.1)(x)
    # x shape: (batch, 3, 16, 16, 128)

    # --- DECODER (Spatial Reconstruction & Fusion) ---
    # Skip S2 fusion
    s2_mean = ReduceMeanTime()(s2)  # FIX: was Lambda (blocked by Keras 3 safe_mode)
    s2_exp  = TileHorizon(horizon)(s2_mean)                  # Expand to Day 8, 9, 10
    x       = layers.Concatenate(axis=-1)([x, s2_exp])
    # x shape: (batch, 3, 16, 16, 256)

    # Upsampling
    x  = layers.TimeDistributed(
        layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu', kernel_regularizer=REG))(x)
    x  = layers.TimeDistributed(
        layers.Cropping2D(((0, 1), (0, 1))))(x)
    # x shape: (batch, 3, 31, 31, 64)

    # Skip S1 fusion
    s1_mean = ReduceMeanTime()(s1)  # FIX: was Lambda (blocked by Keras 3 safe_mode)
    s1_exp  = TileHorizon(horizon)(s1_mean)
    x       = layers.Concatenate(axis=-1)([x, s1_exp])
    # x shape: (batch, 3, 31, 31, 128)

    # Final Polish
    x  = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=REG))(x)
    x  = layers.TimeDistributed(
        layers.BatchNormalization())(x)

    # Output Layer
    out = layers.TimeDistributed(
        layers.Conv2D(1, 1, activation='relu'))(x)
    out = layers.Reshape((horizon, h, w))(out)

    return Model(inp, out)

# ── 4. BUILD & COMPILE ──────────────────────────────────────────────────
print("Building Spatial-Temporal U-Net...")
model = build_cnn_lstm_unet()
model.summary()

Building Spatial-Temporal U-Net...


Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 7, 31, 31, │          0 │ -                 │
│ (InputLayer)        │ 5)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 31, 31, │      1,472 │ input_layer_10[0… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 31, 31, │        128 │ time_distributed… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 31, 31, │     18,496 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 31, 31, │        256 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 16, 16, │          0 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 16, 16, │     73,856 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 7, 16, 16, │        512 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_20      │ (None, 16, 16,    │  1,180,160 │ time_distributed… │
│ (ConvLSTM2D)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 16, 16,    │          0 │ conv_lstm2d_20[0… │
│ (Dropout)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_horizon_10    │ (None, 3, 16, 16, │          0 │ dropout_10[0][0]  │
│ (StackHorizon)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reduce_mean_time_18 │ (None, 16, 16,    │          0 │ time_distributed… │
│ (ReduceMeanTime)    │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_21      │ (None, 3, 16, 16, │  1,180,160 │ stack_horizon_10… │
│ (ConvLSTM2D)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tile_horizon_20     │ (None, 3, 16, 16, │          0 │ reduce_mean_time… │
│ (TileHorizon)       │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_20      │ (None, 3, 16, 16, │          0 │ conv_lstm2d_21[0… │
│ (Concatenate)       │ 256)              │            │ tile_horizon_20[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1… │ (None, 3, 32, 32, │    147,520 │ concatenate_20[0… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reduce_mean_time_19 │ (None, 31, 31,    │          0 │ time_distributed

 Total params: 2,639,617 (10.07 MB)

 Trainable params: 2,639,105 (10.07 MB)

 Non-trainable params: 512 (2.00 KB)

In [29]:
# ══════════════════════════════════════════════════════════════
#  CELL 2 — Load everything in one shot (VS Code / Local)
# ══════════════════════════════════════════════════════════════

import numpy as np
import pickle
import os

# ──   SET YOUR DATA FOLDER HERE ────────────────────────────────
SAVE_DIR = r'C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files'
# ─────────────────────────────────────────────────────────────────

# 1. Load the full feature dataset (needed for lookback window)
print("Loading X_features from .npz (ERA5 weather)...")
dataset    = np.load(os.path.join(SAVE_DIR, 'Mahanadi_DeepLearning_Data.npz'))
X_features = dataset['X']   # (7670, 31, 31, 5)
print(f"   X_features: {X_features.shape}")

# 2. Load y_clean (pre-processed IMD rainfall)
print("Loading y_clean from .npy (IMD rainfall)...")
y_clean = np.load(os.path.join(SAVE_DIR, 'y_clean.npy'))   # (7670, 31, 31)
print(f"   y_clean: {y_clean.shape}")

# 3. Load the fitted scalers
print("Loading scalers...")
with open(os.path.join(SAVE_DIR, 'scaler_X.pkl'), 'rb') as f:
    scaler_X = pickle.load(f)

with open(os.path.join(SAVE_DIR, 'scaler_y.pkl'), 'rb') as f:
    scaler_y = pickle.load(f)

print("   scaler_X and scaler_y loaded.")

# 4. Training split reference
TOTAL_DAYS = 7670
TRAIN_END  = int(TOTAL_DAYS * 0.70)  # 5369
X_train    = X_features[:TRAIN_END]

# 5. scale_array helper
def scale_array(arr, scaler, fit=False):
    if arr.ndim == 4:
        T, H, W, F = arr.shape
        flat = arr.reshape(-1, F)
        if fit: scaler.fit(flat)
        return scaler.transform(flat).reshape(T, H, W, F)
    else:
        T, H, W = arr.shape
        flat = arr.reshape(-1, 1)
        if fit: scaler.fit(flat)
        return scaler.transform(flat).reshape(T, H, W)

print()
print(" All ready — X_features, y_clean, scaler_X, scaler_y, scale_array, X_train")
print("   Run the prediction cell now.")

Loading X_features from .npz (ERA5 weather)...
   X_features: (7670, 31, 31, 5)
Loading y_clean from .npy (IMD rainfall)...
   y_clean: (7670, 31, 31)
Loading scalers...
   scaler_X and scaler_y loaded.

 All ready — X_features, y_clean, scaler_X, scaler_y, scale_array, X_train
   Run the prediction cell now.


c:\Users\ps302\OneDrive\Desktop\Hydrology\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ps302\OneDrive\Desktop\Hydrology\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
# !pip install cdsapi --upgrade -q

# cdsapirc_content = """url: https://cds.climate.copernicus.eu/api
# key: 91c0ca32-abdf-4b66-8530-25a4f1155b4d"""

# with open('/root/.cdsapirc', 'w') as f:
#     f.write(cdsapirc_content)

# print(" .cdsapirc created at /root/.cdsapirc")

In [32]:
def fetch_openmeteo_batched(lat_list, lon_list, batch_size=20):
    all_results = []

    for i in range(0, len(lat_list), batch_size):
        lat_batch = lat_list[i:i+batch_size]
        lon_batch = lon_list[i:i+batch_size]

        print(f"   Fetching batch {i//batch_size + 1} ({len(lat_batch)} points)...")

        for attempt in range(3):  # retry 3 times
            try:
                resp = requests.get(
                    "https://api.open-meteo.com/v1/forecast",
                    params={
                        "latitude": ",".join(map(str, lat_batch)),
                        "longitude": ",".join(map(str, lon_batch)),
                        "hourly": ",".join([
                            "temperature_2m",
                            "pressure_msl",
                            "windspeed_10m",
                            "winddirection_10m",
                            "relativehumidity_850hPa",
                            "temperature_850hPa",
                        ]),
                        "past_days": 5,   #  reduced (was lookback+2)
                        "forecast_days": 1,
                        "wind_speed_unit": "ms",
                        "timezone": "UTC",
                    },
                    timeout=30   #  reduced timeout
                )

                resp.raise_for_status()
                results = resp.json()

                if isinstance(results, dict):
                    results = [results]

                all_results.extend(results)
                break

            except requests.exceptions.RequestException as e:
                print(f"   Retry {attempt+1}/3 failed...")
                if attempt == 2:
                    raise e

    return all_results

In [31]:
# ══════════════════════════════════════════════════════════════════════
#  SMART PREDICTION ENGINE — Final v6
#  WAY 3A now uses Open-Meteo (no CDS, no lag, no API key)
# ══════════════════════════════════════════════════════════════════════
tf.keras.config.enable_unsafe_deserialization()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.interpolate import RegularGridInterpolator
from datetime import date
import os

# ──   ONLY CHANGE THIS LINE ─────────────────────────────────────────
TARGET_START_DATE = '2026-04-06'   # today or any past date
# ──────────────────────────────────────────────────────────────────────
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
MODEL_WEIGHTS    = r'C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files\mahanadi_cnnlstm_unet11.keras'
DATASET_START    = pd.Timestamp('2005-01-01')
TODAY            = pd.Timestamp(date.today())
TOTAL_DAYS       = 7670
TRAIN_END        = int(TOTAL_DAYS * 0.70)
VAL_END          = int(TOTAL_DAYS * 0.85)
DATASET_END      = TOTAL_DAYS - 1
TRAIN_END_DATE   = DATASET_START + pd.Timedelta(days=TRAIN_END - 1)
DATASET_END_DATE = DATASET_START + pd.Timedelta(days=DATASET_END)
LOOKBACK         = 7
HORIZON          = 3
LAT_MIN, LAT_MAX = 17.0, 24.5
LON_MIN, LON_MAX = 80.0, 87.5

target_start       = pd.Timestamp(TARGET_START_DATE)
target_end         = target_start + pd.Timedelta(days=HORIZON - 1)
target_start_idx   = (target_start - DATASET_START).days
target_end_idx     = (target_end   - DATASET_START).days
lookback_start_idx = target_start_idx - (LOOKBACK - 1)
lookback_start     = DATASET_START + pd.Timedelta(days=lookback_start_idx)

# ── DETERMINE WAY ─────────────────────────────────────────────────────
target_in_dataset = (target_start_idx >= 0) and (target_end_idx <= DATASET_END)

if target_start > TODAY:
    WAY = '3B'
elif target_in_dataset and target_end_idx < TRAIN_END:
    WAY = 1
elif target_in_dataset:
    WAY = 2
else:
    WAY = '3A'

# ── WAY 3B HARD BLOCK ─────────────────────────────────────────────────
if WAY == '3B':
    print("=" * 65)
    print(" PREDICTION BLOCKED — FUTURE DATE")
    print("=" * 65)
    print(f"  Target : {target_start.date()} → {target_end.date()}")
    print(f"  Today  : {TODAY.date()}")
    print(f"  No atmospheric data exists for future dates.")
    print(f"  Use TARGET_START_DATE ≤ {TODAY.date()}")
    print("=" * 65)
    raise SystemExit("Blocked: future date.")

WAY_DESCRIPTIONS = {
    1  : " WAY 1  — Stored ERA5 | IMD actual  | validation",
    2  : " WAY 2  — Stored ERA5 | IMD actual  | blind test",
    '3A': " WAY 3A — Open-Meteo live fetch | no lag  | operational",
}

print(f"{'='*65}")
print(f"  Model   : {os.path.basename(MODEL_WEIGHTS)}")
print(f"  Today   : {TODAY.date()}")
print(f"  Target  : {target_start.date()} → {target_end.date()}")
print(f"  Lookback: {lookback_start.date()} → {target_start.date()}")
print(f"{'='*65}")
print(f"  {WAY_DESCRIPTIONS[WAY]}")
print(f"{'='*65}\n")


# ══════════════════════════════════════════════════════════════════════
#  OPEN-METEO HELPERS (WAY 3A)
# ══════════════════════════════════════════════════════════════════════

def rh_to_specific_humidity(rh_percent, temp_celsius, p_hpa=850.0):
    """RH% + T(°C) at pressure level → specific humidity q (kg/kg)."""
    es = 6.112 * np.exp(17.67 * temp_celsius / (temp_celsius + 243.5))
    e  = (rh_percent / 100.0) * es
    return 0.622 * e / (p_hpa - 0.378 * e)


def fetch_openmeteo_for_window(target_date, lookback=7):
    """
    Fetches lookback days of weather ending on target_date.
    Uses Open-Meteo forecast API — zero lag, no API key, free.
    Returns (lookback, 31, 31, 5) matching X_features exactly.
    Channel order: [t2m(K), msl(Pa), q(kg/kg), u10(m/s), v10(m/s)]
    """
    start_date = target_date - pd.Timedelta(days=lookback - 1)
    dates      = [start_date + pd.Timedelta(days=i) for i in range(lookback)]
    date_strs  = [str(d.date()) for d in dates]
    print(f"   Window: {date_strs[0]} → {date_strs[-1]}")

    # Sparse 9×9 fetch grid → interpolate to dense 31×31
    N         = 9
    S_LATS    = np.linspace(17.0, 24.5, N)
    S_LONS    = np.linspace(80.0, 87.5, N)
    D_LATS    = np.linspace(17.0, 24.5, 31)
    D_LONS    = np.linspace(80.0, 87.5, 31)

    lat_list  = [round(float(la), 4) for la in S_LATS for _ in S_LONS]
    lon_list  = [round(float(lo), 4) for _ in S_LATS for lo in S_LONS]

    print(f"   Calling Open-Meteo API for {len(lat_list)} grid points...")

    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude"               : ",".join(map(str, lat_list)),
            "longitude"              : ",".join(map(str, lon_list)),
            "hourly"                 : ",".join([
                "temperature_2m",
                "pressure_msl",
                "windspeed_10m",
                "winddirection_10m",
                "relativehumidity_850hPa",
                "temperature_850hPa",
            ]),
            "past_days"              : lookback + 2,
            "forecast_days"          : 1,
            "wind_speed_unit"        : "ms",
            "timezone"               : "UTC",
        },
        timeout=120
    )
    resp.raise_for_status()
    results = resp.json()
    if isinstance(results, dict):
        results = [results]

    print(f"   Received {len(results)} locations ")

    # ── Parse into sparse array ────────────────────────────────────────
    sparse = np.zeros((N, N, lookback, 5))

    for idx, res in enumerate(results):
        i      = idx // N
        j      = idx %  N
        hourly = res['hourly']
        times  = pd.DatetimeIndex(hourly['time'])

        for k, d in enumerate(dates):
            # Find 12:00 UTC for this date
            target_dt  = pd.Timestamp(d.date()).replace(hour=12)
            time_diffs = np.abs((times - target_dt).total_seconds())
            t_idx      = int(np.argmin(time_diffs))

            def safe(val, default):
                return float(val) if val is not None else default

            t2m_c  = safe(hourly['temperature_2m'][t_idx],           25.0)
            pmsl   = safe(hourly['pressure_msl'][t_idx],           1013.0)
            wspd   = safe(hourly['windspeed_10m'][t_idx],              0.0)
            wdir   = safe(hourly['winddirection_10m'][t_idx],          0.0)
            rh850  = safe(hourly['relativehumidity_850hPa'][t_idx],   60.0)
            t850_c = safe(hourly['temperature_850hPa'][t_idx],        15.0)

            # Convert to ERA5 units
            t2m_k    = t2m_c + 273.15          # °C → K
            msl_pa   = pmsl  * 100.0           # hPa → Pa
            q        = rh_to_specific_humidity(rh850, t850_c)
            wdir_rad = np.deg2rad(wdir)
            u10      = -wspd * np.sin(wdir_rad)
            v10      = -wspd * np.cos(wdir_rad)

            sparse[i, j, k] = [t2m_k, msl_pa, q, u10, v10]

    # ── Interpolate 9×9 → 31×31 ───────────────────────────────────────
    X_window   = np.zeros((lookback, 31, 31, 5))
    lat_g, lon_g = np.meshgrid(D_LATS, D_LONS, indexing='ij')
    pts          = np.stack([lat_g.ravel(), lon_g.ravel()], axis=-1)

    for k in range(lookback):
        for v in range(5):
            fn = RegularGridInterpolator(
                (S_LATS, S_LONS), sparse[:, :, k, v],
                method='linear', bounds_error=False, fill_value=None
            )
            X_window[k, :, :, v] = fn(pts).reshape(31, 31)

    print(f" Open-Meteo fetch complete — shape: {X_window.shape}")
    names = ['t2m(K)', 'msl(Pa)', 'q(kg/kg)', 'u10(m/s)', 'v10(m/s)']
    for v, nm in enumerate(names):
        print(f"     {nm:<12} basin-mean: {X_window[:, :, :, v].mean():.4f}")

    return X_window


# ══════════════════════════════════════════════════════════════════════
#  STEP 1: LOAD MODEL
# ══════════════════════════════════════════════════════════════════════
# Keras 3 note: this .keras archive contains legacy Lambda layers.
# To avoid deserialization failures, rebuild architecture and load weights.
import tensorflow as tf
import zipfile
import tempfile

def load_model_weights_robust(weights_path):
    model = build_cnn_lstm_unet(
        lookback=LOOKBACK, h=31, w=31, n_feats=5, horizon=HORIZON
    )

    if str(weights_path).lower().endswith('.keras'):
        with zipfile.ZipFile(weights_path, 'r') as zf:
            names = set(zf.namelist())
            if 'model.weights.h5' not in names:
                raise FileNotFoundError(
                    'model.weights.h5 not found inside .keras archive.'
                )
            tmp_dir = tempfile.mkdtemp(prefix='keras_weights_')
            zf.extract('model.weights.h5', path=tmp_dir)
            h5_path = os.path.join(tmp_dir, 'model.weights.h5')
            model.load_weights(h5_path)
    else:
        model.load_weights(weights_path)

    return model

model = load_model_weights_robust(MODEL_WEIGHTS)
print(" Model architecture rebuilt and weights loaded.\n")

# ══════════════════════════════════════════════════════════════════════
#  STEP 2: BUILD LOOKBACK INPUT
# ══════════════════════════════════════════════════════════════════════
if WAY in (1, 2):
    # Stored ERA5 in X_features
    X_window_raw    = X_features[lookback_start_idx : lookback_start_idx + LOOKBACK]
    X_window_scaled = scale_array(X_window_raw, scaler_X, fit=False)
    print(f" Lookback from stored X_features: {lookback_start.date()} → {target_start.date()}")

else:  # WAY 3A — Open-Meteo
    print(" WAY 3A: Fetching live data from Open-Meteo...")
    X_window_raw    = fetch_openmeteo_for_window(target_start, LOOKBACK)
    X_window_scaled = scale_array(X_window_raw, scaler_X, fit=False)
    print(" Live data fetched and scaled.\n")

X_input = X_window_scaled[np.newaxis, ...]    # (1, 7, 31, 31, 5)

# ══════════════════════════════════════════════════════════════════════
#  STEP 3: PREDICT
# ══════════════════════════════════════════════════════════════════════
predicted_scaled = model.predict(X_input, verbose=0)
predicted_scaled = np.clip(predicted_scaled, 0, None)

def unscale(arr, scaler):
    return scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)

pred_mm = unscale(predicted_scaled[0], scaler_y)   # (3, 31, 31) mm
print(" Prediction complete.\n")

# ══════════════════════════════════════════════════════════════════════
#  STEP 4: ACTUAL IMD (WAY 1 & 2 only)
# ══════════════════════════════════════════════════════════════════════
has_actual = False
if WAY in (1, 2):
    actual_raw = y_clean[target_start_idx : target_start_idx + HORIZON]
    if actual_raw.shape[0] == HORIZON:
        actual_mm  = actual_raw
        has_actual = True
        print(f" Actual IMD data loaded: {target_start.date()} → {target_end.date()}\n")

# ══════════════════════════════════════════════════════════════════════
#  STEP 5: PLOT
# ══════════════════════════════════════════════════════════════════════
n_rows     = 2 if has_actual else 1
row_labels = ['ACTUAL (IMD)', 'PREDICTED'] if has_actual else ['PREDICTED']
data_rows  = [actual_mm, pred_mm]          if has_actual else [pred_mm]

way_label = {
    1  : "Way 1 — Training data (validation)",
    2  : "Way 2 — Blind Val/Test (real evaluation)",
    '3A': "Way 3A — Open-Meteo live fetch (operational forecast)",
}

fig, axes = plt.subplots(n_rows, HORIZON,
                          figsize=(6 * HORIZON, 5 * n_rows), squeeze=False)
fig.suptitle(
    f"Mahanadi Basin: 3-Day Rainfall Forecast\n"
    f"{target_start.date()} → {target_end.date()}   |   {way_label[WAY]}",
    fontsize=13, fontweight='bold'
)

for row_i, (label, maps) in enumerate(zip(row_labels, data_rows)):
    for d in range(HORIZON):
        ax       = axes[row_i, d]
        day_date = target_start + pd.Timedelta(days=d)
        vmax     = max(
            float(actual_mm[d].max()) if has_actual else 0.0,
            float(pred_mm[d].max()), 1.0
        )
        im = ax.imshow(maps[d], cmap='YlGnBu', vmin=0, vmax=vmax,
                       origin='lower', extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
        ax.set_title(f"{label}\n{day_date.strftime('%d %b %Y')}", fontsize=11)
        ax.set_xlabel("Longitude (°E)", fontsize=9)
        ax.set_ylabel("Latitude (°N)",  fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mm')
        if has_actual and label == 'PREDICTED':
            mae = float(np.mean(np.abs(actual_mm[d] - pred_mm[d])))
            ax.set_xlabel(f"Day {d+1} MAE = {mae:.2f} mm", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

# ══════════════════════════════════════════════════════════════════════
#  STEP 6: SUMMARY
# ══════════════════════════════════════════════════════════════════════
print(" Basin-average results:")
print(f"  {'Date':<14} {'Pred mean':>10} {'Pred max':>10}"
      + (f" {'Actual mean':>12} {'Day MAE':>10}" if has_actual else ""))
print("  " + "─" * (46 + (24 if has_actual else 0)))

for d in range(HORIZON):
    day_date = target_start + pd.Timedelta(days=d)
    pm = float(np.mean(pred_mm[d]))
    px = float(np.max(pred_mm[d]))
    line = f"  {str(day_date.date()):<14} {pm:>10.2f} {px:>10.2f}"
    if has_actual:
        am  = float(np.mean(actual_mm[d]))
        mae = float(np.mean(np.abs(actual_mm[d] - pred_mm[d])))
        line += f" {am:>12.2f} {mae:>10.2f}"
    print(line)

if has_actual:
    print(f"\n  Overall 3-day MAE: {float(np.mean(np.abs(actual_mm - pred_mm))):.2f} mm")

if WAY == '3A':
    print(f"\n   Source : Open-Meteo (ECMWF IFS + ERA5 reanalysis blend)")
    print(f"   Lag    : Zero — data available up to current hour")
    print(f"   Note   : Small distribution shift vs ERA5 training data is expected")
    print(f"              but values are meteorologically equivalent.")

  Model   : mahanadi_cnnlstm_unet11.keras
  Today   : 2026-04-07
  Target  : 2026-04-06 → 2026-04-08
  Lookback: 2026-03-31 → 2026-04-06
   WAY 3A — Open-Meteo live fetch | no lag  | operational

 Model architecture rebuilt and weights loaded.

 WAY 3A: Fetching live data from Open-Meteo...
   Window: 2026-03-31 → 2026-04-06
   Calling Open-Meteo API for 81 grid points...


ReadTimeout: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=120)

In [33]:
tf.keras.config.enable_unsafe_deserialization()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.interpolate import RegularGridInterpolator
from datetime import date
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

TARGET_START_DATE = '2026-04-06'

MODEL_WEIGHTS    = r'C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files\mahanadi_cnnlstm_unet11.keras'
DATASET_START    = pd.Timestamp('2005-01-01')
TODAY            = pd.Timestamp(date.today())

TOTAL_DAYS       = 7670
TRAIN_END        = int(TOTAL_DAYS * 0.70)
DATASET_END      = TOTAL_DAYS - 1

LOOKBACK = 7
HORIZON  = 3

LAT_MIN, LAT_MAX = 17.0, 24.5
LON_MIN, LON_MAX = 80.0, 87.5

target_start = pd.Timestamp(TARGET_START_DATE)
target_end   = target_start + pd.Timedelta(days=HORIZON - 1)

target_start_idx   = (target_start - DATASET_START).days
target_end_idx     = (target_end   - DATASET_START).days
lookback_start_idx = target_start_idx - (LOOKBACK - 1)

target_in_dataset = (target_start_idx >= 0) and (target_end_idx <= DATASET_END)

if target_start > TODAY:
    raise SystemExit("Future date not allowed")

elif target_in_dataset and target_end_idx < TRAIN_END:
    WAY = 1
elif target_in_dataset:
    WAY = 2
else:
    WAY = '3A'

print(f"WAY SELECTED: {WAY}")

WAY SELECTED: 3A


In [ ]:
def rh_to_specific_humidity(rh_percent, temp_celsius, p_hpa=850.0):
    es = 6.112 * np.exp(17.67 * temp_celsius / (temp_celsius + 243.5))
    e  = (rh_percent / 100.0) * es
    return 0.622 * e / (p_hpa - 0.378 * e)


#  NEW: batched API
def fetch_openmeteo_batched(lat_list, lon_list, batch_size=20):
    all_results = []

    for i in range(0, len(lat_list), batch_size):
        lat_batch = lat_list[i:i+batch_size]
        lon_batch = lon_list[i:i+batch_size]

        for attempt in range(3):
            try:
                resp = requests.get(
                    "https://api.open-meteo.com/v1/forecast",
                    params={
                        "latitude": ",".join(map(str, lat_batch)),
                        "longitude": ",".join(map(str, lon_batch)),
                        "hourly": "temperature_2m,pressure_msl,windspeed_10m,winddirection_10m,relativehumidity_850hPa,temperature_850hPa",
                        "past_days": 5,
                        "forecast_days": 1,
                        "timezone": "UTC"
                    },
                    timeout=30
                )

                resp.raise_for_status()
                results = resp.json()
                if isinstance(results, dict):
                    results = [results]

                all_results.extend(results)
                break

            except:
                if attempt == 2:
                    raise

    return all_results


def fetch_openmeteo_for_window(target_date, lookback=7):

    start_date = target_date - pd.Timedelta(days=lookback - 1)
    dates = [start_date + pd.Timedelta(days=i) for i in range(lookback)]

    #  Reduced grid (fast)
    N = 7
    S_LATS = np.linspace(17.0, 24.5, N)
    S_LONS = np.linspace(80.0, 87.5, N)

    D_LATS = np.linspace(17.0, 24.5, 31)
    D_LONS = np.linspace(80.0, 87.5, 31)

    lat_list = [la for la in S_LATS for _ in S_LONS]
    lon_list = [lo for _ in S_LATS for lo in S_LONS]

    print(f"Fetching {len(lat_list)} points (batched)...")

    results = fetch_openmeteo_batched(lat_list, lon_list)

    sparse = np.zeros((N, N, lookback, 5))

    for idx, res in enumerate(results):
        i, j = idx // N, idx % N
        hourly = res['hourly']
        times = pd.DatetimeIndex(hourly['time'])

        for k, d in enumerate(dates):
            target_dt = pd.Timestamp(d.date()).replace(hour=12)
            t_idx = np.argmin(np.abs(times - target_dt))

            t2m_c  = hourly['temperature_2m'][t_idx]
            pmsl   = hourly['pressure_msl'][t_idx]
            wspd   = hourly['windspeed_10m'][t_idx]
            wdir   = hourly['winddirection_10m'][t_idx]
            rh850  = hourly['relativehumidity_850hPa'][t_idx]
            t850_c = hourly['temperature_850hPa'][t_idx]

            t2m_k  = t2m_c + 273.15
            msl_pa = pmsl * 100.0
            q      = rh_to_specific_humidity(rh850, t850_c)

            wdir_rad = np.deg2rad(wdir)
            u10 = -wspd * np.sin(wdir_rad)
            v10 = -wspd * np.cos(wdir_rad)

            sparse[i, j, k] = [t2m_k, msl_pa, q, u10, v10]

    # interpolation
    X_window = np.zeros((lookback, 31, 31, 5))
    lat_g, lon_g = np.meshgrid(D_LATS, D_LONS, indexing='ij')
    pts = np.stack([lat_g.ravel(), lon_g.ravel()], axis=-1)

    for k in range(lookback):
        for v in range(5):
            fn = RegularGridInterpolator((S_LATS, S_LONS), sparse[:, :, k, v])
            X_window[k, :, :, v] = fn(pts).reshape(31, 31)

    return X_window

In [35]:
import tensorflow as tf
import zipfile
import tempfile

def load_model_weights_robust(weights_path):
    model = build_cnn_lstm_unet(
        lookback=LOOKBACK, h=31, w=31, n_feats=5, horizon=HORIZON
    )

    with zipfile.ZipFile(weights_path, 'r') as zf:
        tmp_dir = tempfile.mkdtemp()
        zf.extract('model.weights.h5', path=tmp_dir)
        model.load_weights(os.path.join(tmp_dir, 'model.weights.h5'))

    return model


model = load_model_weights_robust(MODEL_WEIGHTS)

# INPUT
if WAY == '3A':
    X_window_raw = fetch_openmeteo_for_window(target_start, LOOKBACK)
    X_window_scaled = scale_array(X_window_raw, scaler_X, fit=False)
else:
    X_window_scaled = scale_array(
        X_features[lookback_start_idx:lookback_start_idx + LOOKBACK],
        scaler_X,
        fit=False
    )

X_input = X_window_scaled[np.newaxis, ...]

# PREDICT
pred_scaled = model.predict(X_input)
pred_mm = scaler_y.inverse_transform(pred_scaled.reshape(-1,1)).reshape(pred_scaled.shape)

print("Prediction done:", pred_mm.shape)

Fetching 49 points (batched)...


SSLError: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Max retries exceeded with url: /v1/forecast?latitude=17.0%2C17.0%2C17.0%2C17.0%2C17.0%2C17.0%2C17.0%2C18.25%2C18.25%2C18.25%2C18.25%2C18.25%2C18.25%2C18.25%2C19.5%2C19.5%2C19.5%2C19.5%2C19.5%2C19.5&longitude=80.0%2C81.25%2C82.5%2C83.75%2C85.0%2C86.25%2C87.5%2C80.0%2C81.25%2C82.5%2C83.75%2C85.0%2C86.25%2C87.5%2C80.0%2C81.25%2C82.5%2C83.75%2C85.0%2C86.25&hourly=temperature_2m%2Cpressure_msl%2Cwindspeed_10m%2Cwinddirection_10m%2Crelativehumidity_850hPa%2Ctemperature_850hPa&past_days=5&forecast_days=1&timezone=UTC (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))

In [26]:
def predict_at_location(lat, lon):
    """
    Given a lat/lon:
    1. Finds the nearest grid point
    2. Selects all pixels within ±1 degree of that nearest point
    3. For each of the 3 forecast days, finds the pixel with MAX predicted
       rainfall inside that rectangle and reports its coordinates,
       along with max pred, mean pred, and max actual at that same pixel.
    """

    LATS = np.linspace(17.0, 24.5, 31)
    LONS = np.linspace(80.0, 87.5, 31)

    # ── Step 1: nearest grid point ───────────────────────────────
    nearest_row = int(np.argmin(np.abs(LATS - lat)))
    nearest_col = int(np.argmin(np.abs(LONS - lon)))
    nearest_lat = LATS[nearest_row]
    nearest_lon = LONS[nearest_col]

    # ── Step 2: all row/col indices within ±1 degree ─────────────
    row_indices = np.where(np.abs(LATS - nearest_lat) <= 1.0)[0]
    col_indices = np.where(np.abs(LONS - nearest_lon) <= 1.0)[0]

    # Sub-grid slice boundaries (for efficient indexing)
    r_min, r_max = row_indices[0],  row_indices[-1]
    c_min, c_max = col_indices[0],  col_indices[-1]

    n_pixels = len(row_indices) * len(col_indices)

    # ── Step 3: Print header ─────────────────────────────────────
    print(f"Query     : {lat:.2f}°N, {lon:.2f}°E")
    print(f"Nearest   : {nearest_lat:.2f}°N, {nearest_lon:.2f}°E")
    print(f"Area      : {LATS[r_min]:.2f}–{LATS[r_max]:.2f}°N, "
          f"{LONS[c_min]:.2f}–{LONS[c_max]:.2f}°E  ({n_pixels} pixels)")
    print()

    if has_actual:
        print(f"  {'Date':<14} {'Max coord':>18} {'Max pred':>10} "
              f"{'Mean pred':>10} {'Actual @ max':>13}")
        print(f"  {'─'*70}")
    else:
        print(f"  {'Date':<14} {'Max coord':>18} {'Max pred':>10} {'Mean pred':>10}")
        print(f"  {'─'*57}")

    # ── Step 4: Per-day analysis ─────────────────────────────────
    for d in range(HORIZON):
        day_date = target_start + pd.Timedelta(days=d)

        # Extract the sub-grid patch for predicted rainfall
        pred_patch = pred_mm[d, r_min:r_max+1, c_min:c_max+1]
        # shape: (n_rows_in_area, n_cols_in_area)

        # Find position of max predicted value within the patch
        flat_idx        = int(np.argmax(pred_patch))
        local_row, local_col = np.unravel_index(flat_idx, pred_patch.shape)

        # Convert back to global grid indices
        global_row = r_min + local_row
        global_col = c_min + local_col

        # Real-world coordinates of the max pixel
        max_lat = LATS[global_row]
        max_lon = LONS[global_col]

        max_pred  = float(pred_patch[local_row, local_col])
        mean_pred = float(pred_patch.mean())

        coord_str = f"({max_lat:.2f}°N,{max_lon:.2f}°E)"

        if has_actual:
            # Actual value at the SAME pixel that had max predicted rainfall
            actual_at_max = float(actual_mm[d, global_row, global_col])
            print(f"  {str(day_date.date()):<14} {coord_str:>18} {max_pred:>10.2f} "
                  f"{mean_pred:>10.2f} {actual_at_max:>12.2f}")
        else:
            print(f"  {str(day_date.date()):<14} {coord_str:>18} {max_pred:>10.2f} "
                  f"{mean_pred:>10.2f}")

    print()


# ── Call it ──────────────────────────────────────────────────────
predict_at_location(21.47064, 83.98369)
predict_at_location(20.45275, 85.81421)
# predict_at_location(19.0, 85.0)

Query     : 21.47°N, 83.98°E
Nearest   : 21.50°N, 84.00°E
Area      : 20.50–22.50°N, 83.00–85.00°E  (81 pixels)

  Date                    Max coord   Max pred  Mean pred  Actual @ max
  ──────────────────────────────────────────────────────────────────────
  2025-04-06      (20.50°N,83.00°E)       0.00       0.00         0.00
  2025-04-07      (20.50°N,83.00°E)       0.00       0.00         0.00
  2025-04-08      (22.00°N,85.00°E)       0.27       0.01         0.00

Query     : 20.45°N, 85.81°E
Nearest   : 20.50°N, 85.75°E
Area      : 19.50–21.50°N, 84.75–86.75°E  (81 pixels)

  Date                    Max coord   Max pred  Mean pred  Actual @ max
  ──────────────────────────────────────────────────────────────────────
  2025-04-06      (19.50°N,84.75°E)       0.00       0.00         0.00
  2025-04-07      (21.50°N,86.75°E)       0.86       0.01         0.00
  2025-04-08      (21.50°N,86.75°E)       1.31       0.06         0.00



In [22]:
# Quick check: notebook-side values at app location
lat_chk, lon_chk = 21.47064, 83.98369
LATS = np.linspace(17.0, 24.5, 31)
LONS = np.linspace(80.0, 87.5, 31)
r = int(np.argmin(np.abs(LATS - lat_chk)))
c = int(np.argmin(np.abs(LONS - lon_chk)))
print("nearest grid:", LATS[r], LONS[c])
for d in range(HORIZON):
    day_date = target_start + pd.Timedelta(days=d)
    print(
        str(day_date.date()),
        round(float(pred_mm[d, r, c]), 9),
        round(float(np.mean(pred_mm[d])), 9),
        round(float(np.max(pred_mm[d])), 9),
    )

nearest grid: 21.5 84.0
2026-04-06 3.854523182 2.554843426 8.623200417
2026-04-07 3.735910177 2.842758656 6.841254711
2026-04-08 3.638601542 2.855441809 6.82631588
